In [1]:
import os
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter
import pandas as pd

index_personnes = Path("../../../corpus/IndexPersonnes.xml")

ns = "{http://www.tei-c.org/ns/1.0}"
xml_ns = "{http://www.w3.org/XML/1998/namespace}"

data = []

# Parsing en streaming
context = ET.iterparse(index_personnes, events=("end",))

for event, elem in context:
    if elem.tag == f"{ns}person":
        
        xml_id = elem.get(f"{xml_ns}id")
        role = elem.get("role")
        source = elem.get("source")

        # naissance
        birth = elem.find(f"{ns}birth")
        birth_date = birth.get("when") if birth is not None else None
        birth_place = birth.find(f"{ns}placeName").text if (birth is not None and birth.find(f"{ns}placeName") is not None) else None

        # mort
        death = elem.find(f"{ns}death")
        death_date = death.get("when") if death is not None else None
        death_place = death.find(f"{ns}placeName").text if (death is not None and death.find(f"{ns}placeName") is not None) else None

        data.append({
            "xml_id": xml_id,
            "role": role,
            "wikidata_source": source,
            "birth_date": birth_date,
            "birth_place": birth_place,
            "death_date": death_date,
            "death_place": death_place
        })

        # 🔥 TRÈS IMPORTANT : libérer la mémoire
        elem.clear()

# DataFrame
df = pd.DataFrame(data)

print(df.head())

df.to_csv("personnes.csv", index=False)

FileNotFoundError: [Errno 2] No such file or directory: '../../../corpus/IndexPersonnes.xml'

In [ ]:
import os
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter
import pandas as pd
import time


index_personnes = Path("../../../corpus/IndexPersonnes.xml")
index_lieux = Path("../../../corpus/IndexPersonnes.xml")



In [ ]:
import xml.etree.ElementTree as ET
import os
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter
import pandas as pd
import requests

# Namespaces
TEI_NS = "{http://www.tei-c.org/ns/1.0}"
XML_NS = "{http://www.w3.org/XML/1998/namespace}"

# Endpoint SPARQL Wikidata
SPARQL_URL = "https://query.wikidata.org/sparql"

headers = {
    "User-Agent": "TEI-Wikidata-Script/1.0 (contact@example.com)"
}

# ----------------------------
# 1. EXTRAIRE LES QID PERSONNES
# ----------------------------
def extract_person_qids(xml_file):
    qids = []

    context = ET.iterparse(xml_file, events=("end",))
    for event, elem in context:
        if elem.tag == f"{TEI_NS}person":
            source = elem.get("source")
            if source and "wikidata.org" in source:
                qid = source.split("/")[-1]
                qids.append(qid)
            elem.clear()

    return list(set(qids))  # unique


# ----------------------------
# 2. SPARQL → LIEUX P19 / P20
# ----------------------------
def get_places_from_wikidata(qid):
    query = f"""
    SELECT ?birthPlace ?deathPlace WHERE {{
      OPTIONAL {{ wd:{qid} wdt:P19 ?birthPlace. }}
      OPTIONAL {{ wd:{qid} wdt:P20 ?deathPlace. }}
    }}
    """

    response = requests.get(
        SPARQL_URL,
        params={"query": query, "format": "json"},
        headers=headers
    )

    data = response.json()

    places = []

    for result in data["results"]["bindings"]:
        if "birthPlace" in result:
            places.append(result["birthPlace"]["value"].split("/")[-1])
        if "deathPlace" in result:
            places.append(result["deathPlace"]["value"].split("/")[-1])

    return places


# ----------------------------
# 3. EXTRAIRE INDEX DES LIEUX
# ----------------------------
def extract_place_index(xml_file):
    place_ids = set()

    context = ET.iterparse(xml_file, events=("end",))
    for event, elem in context:
        if elem.tag == f"{TEI_NS}place":
            source = elem.get("source")
            if source and "wikidata.org" in source:
                qid = source.split("/")[-1]
                place_ids.add(qid)
            elem.clear()

    return place_ids


# ----------------------------
# 4. PIPELINE PRINCIPAL
# ----------------------------
def main(person_xml, place_xml):

    print("Extraction des personnes...")
    person_qids = extract_person_qids(person_xml)

    print(f"{len(person_qids)} personnes trouvées")

    all_places = set()

    print("Requête Wikidata...")
    for i, qid in enumerate(person_qids):
        try:
            places = get_places_from_wikidata(qid)
            all_places.update(places)

            # ⚠️ éviter blocage Wikidata (rate limit)
            time.sleep(0.1)

        except Exception as e:
            print(f"Erreur avec {qid}: {e}")

    print(f"{len(all_places)} lieux récupérés depuis Wikidata")

    # Sauvegarde TXT
    with open("wikidata_places.txt", "w", encoding="utf-8") as f:
        for p in sorted(all_places):
            f.write(p + "\n")

    print("Extraction index des lieux TEI...")
    place_index = extract_place_index(place_xml)

    # Comparaison
    missing = all_places - place_index

    print(f"{len(missing)} lieux manquants dans l'index TEI")

    with open("missing_places.txt", "w", encoding="utf-8") as f:
        for m in sorted(missing):
            f.write(m + "\n")

index_personnes = Path("../../corpus/IndexPersonnes.xml")
index_lieux = Path("../../../corpus/IndexPersonnes.xml")

# ----------------------------
# LANCEMENT
# ----------------------------
if __name__ == "__main__":
    main(index_personnes, index_lieux)

Extraction des personnes...
727 personnes trouvées
Requête Wikidata...
Erreur avec Q380537: name 'time' is not defined
Erreur avec Q5664: name 'time' is not defined
Erreur avec Q822909: name 'time' is not defined
Erreur avec Q334188: name 'time' is not defined
Erreur avec Q222923: name 'time' is not defined
Erreur avec Q8053: name 'time' is not defined
Erreur avec Q535474: name 'time' is not defined
Erreur avec Q320980: name 'time' is not defined
Erreur avec Q286685: name 'time' is not defined
Erreur avec Q1374872: name 'time' is not defined
Erreur avec Q673573: name 'time' is not defined
Erreur avec Q314548: name 'time' is not defined
Erreur avec Q601175: name 'time' is not defined
Erreur avec Q3340535: name 'time' is not defined
Erreur avec Q1160847: name 'time' is not defined
Erreur avec Q469379: name 'time' is not defined
Erreur avec Q544264: name 'time' is not defined
Erreur avec Q982274: name 'time' is not defined
Erreur avec Q2623716: name 'time' is not defined
Erreur avec Q3960

KeyboardInterrupt: 